# Eksperimen 19 — Main-Ingredient-Aware Efficiency (Validation Only)

Eksperimen ini hanya menggunakan **validation v2 (38 query)**.

Tujuan:
- mempertahankan arsitektur utama **SBERT + Weighted Linear Combination (WLC)**;
- memperbaiki komponen ingredient efficiency dengan memberi perhatian khusus pada kecocokan **bahan utama**;
- membandingkan hasil terhadap baseline lama.

Baseline:
- Retrieval: semantic Top-50 + keyword tambahan maksimal 20
- Semantic normalization: min-max
- WLC: 0.7 semantic + 0.3 harmonic efficiency

Efficiency baru:

\[
E' = lpha E_{harmonic} + (1-lpha) C_{main}
\]

dengan:

\[
C_{main} =
rac{	ext{jumlah bahan utama user yang cocok pada resep}}
{	ext{jumlah bahan utama user}}
\]

Jika query tidak memiliki bahan utama terklasifikasi, `C_main` dibuat fallback ke `coverage_user`
agar skor tetap terdefinisi.

Eksperimen hanya memakai:
- query user,
- ingredient kandidat,
- klasifikasi bahan utama yang sudah tersedia pada sistem.

Ground truth hanya digunakan untuk evaluasi, bukan untuk menghitung score.


In [1]:
import pandas as pd
import numpy as np

from dapuria_pipeline import (
    DapuriaRecipeService,
    parse_user_ingredients,
    coverage_stats,
    harmonic_efficiency,
    ingredient_matches,
    is_main_ingredient,
)

VAL_PATH = "ground_truth_relevance_validation_v2.csv"

SUMMARY_OUT = "hasil_main_ingredient_aware_validation_ringkas.csv"
DETAIL_OUT = "hasil_main_ingredient_aware_validation_detail.csv"


def split_pipe(x):
    if pd.isna(x):
        return []
    return [s.strip() for s in str(x).split(" | ") if s.strip()]


def hit_at_k(ranked_ids, relevant_ids, k):
    rel = set(relevant_ids)
    return int(any(rid in rel for rid in ranked_ids[:k]))


def precision_at_k(ranked_ids, relevant_ids, k):
    rel = set(relevant_ids)
    return sum(rid in rel for rid in ranked_ids[:k]) / k


def recall_at_k(ranked_ids, relevant_ids, k):
    rel = set(relevant_ids)
    if not rel:
        return 0.0
    return sum(rid in rel for rid in ranked_ids[:k]) / len(rel)


def reciprocal_rank(ranked_ids, relevant_ids):
    rel = set(relevant_ids)
    for rank, rid in enumerate(ranked_ids, start=1):
        if rid in rel:
            return 1.0 / rank
    return 0.0


def minmax(values):
    arr = np.asarray(values, dtype=float)
    if len(arr) == 0:
        return arr
    lo = float(arr.min())
    hi = float(arr.max())
    return (arr - lo) / (hi - lo + 1e-9)


def main_ingredient_coverage(user_ingredients, recipe_ingredients):
    """
    Proporsi bahan utama user yang cocok pada kandidat.

    Jika tidak ada bahan user yang terklasifikasi sebagai bahan utama,
    fallback ke coverage_user biasa agar score tetap terdefinisi.
    """
    main_user = [u for u in user_ingredients if is_main_ingredient(u)]

    if main_user:
        matched = 0
        for u in main_user:
            if any(ingredient_matches(u, r) for r in recipe_ingredients):
                matched += 1

        return matched / len(main_user)

    # fallback
    stats = coverage_stats(
        user_ingredients,
        recipe_ingredients,
    )
    return float(stats["coverage_user"])


# ============================================================
# 1. Load validation set
# ============================================================
val = pd.read_csv(VAL_PATH)

required = {
    "scraped_title",
    "query_text_v2",
    "relevant_ids",
}

missing = required - set(val.columns)

if missing:
    raise ValueError(f"Kolom validation hilang: {sorted(missing)}")

dup = val.duplicated(["query_text_v2", "relevant_ids"]).sum()

if dup:
    raise ValueError(f"Validation masih punya {dup} duplicate cases.")

print("VALIDASI DATA")
print(f"Jumlah query              : {len(val)}")
print(f"Duplicate query+relevance : {dup}")


# ============================================================
# 2. Init service
# ============================================================
service = DapuriaRecipeService(
    chroma_dir="chroma_db",
    collection_name="resep_masakan",
)

print("\nBASE RETRIEVAL CONFIG")
print("Semantic Top-K            : 50")
print("Keyword extra max         : 20")
print("Semantic normalization    : min-max")


# ============================================================
# 3. Cache candidates once
# ============================================================
cases = []

print("\nMembangun candidate pool validation...")

for _, row in val.iterrows():
    query_text = str(row["query_text_v2"]).strip()
    relevant_ids = set(split_pipe(row["relevant_ids"]))

    user_ingredients = parse_user_ingredients(query_text)

    candidates = service.retrieve_candidates(
        query_text,
        user_ingredients,
    )

    semantic_mm = minmax([float(c["sbert_score"]) for c in candidates])

    prepared = []

    for c, s_norm in zip(
        candidates,
        semantic_mm,
    ):
        stats = coverage_stats(
            user_ingredients,
            c["ingredient_names"],
        )

        eff_harmonic = harmonic_efficiency(
            stats["coverage_recipe"],
            stats["coverage_user"],
        )

        main_cov = main_ingredient_coverage(
            user_ingredients,
            c["ingredient_names"],
        )

        prepared.append(
            {
                **c,
                "semantic_minmax": float(s_norm),
                "coverage_recipe": float(stats["coverage_recipe"]),
                "coverage_user": float(stats["coverage_user"]),
                "eff_harmonic": float(eff_harmonic),
                "main_coverage": float(main_cov),
            }
        )

    cases.append(
        {
            "scraped_title": row["scraped_title"],
            "query_text": query_text,
            "relevant_ids": relevant_ids,
            "user_ingredients": user_ingredients,
            "candidates": prepared,
        }
    )

print("Candidate cache selesai.")


# ============================================================
# 4. Configurations
# ============================================================
configs = [
    {
        "name": "BASELINE_07_03",
        "alpha_harmonic": 1.00,
        "w_semantic": 0.70,
        "w_eff": 0.30,
    },
    # E' = 0.75 harmonic + 0.25 main coverage
    {
        "name": "MAIN075_W07_03",
        "alpha_harmonic": 0.75,
        "w_semantic": 0.70,
        "w_eff": 0.30,
    },
    {
        "name": "MAIN075_W06_04",
        "alpha_harmonic": 0.75,
        "w_semantic": 0.60,
        "w_eff": 0.40,
    },
    # E' = 0.50 harmonic + 0.50 main coverage
    {
        "name": "MAIN050_W07_03",
        "alpha_harmonic": 0.50,
        "w_semantic": 0.70,
        "w_eff": 0.30,
    },
    {
        "name": "MAIN050_W06_04",
        "alpha_harmonic": 0.50,
        "w_semantic": 0.60,
        "w_eff": 0.40,
    },
    # E' = 0.25 harmonic + 0.75 main coverage
    {
        "name": "MAIN025_W07_03",
        "alpha_harmonic": 0.25,
        "w_semantic": 0.70,
        "w_eff": 0.30,
    },
    {
        "name": "MAIN025_W06_04",
        "alpha_harmonic": 0.25,
        "w_semantic": 0.60,
        "w_eff": 0.40,
    },
]


# ============================================================
# 5. Evaluate configs
# ============================================================
summary_rows = []
detail_rows = []

for config in configs:
    candidate_hits = []
    h5s = []
    h10s = []
    p5s = []
    p10s = []
    r5s = []
    r10s = []
    rrs = []

    for case in cases:
        candidates = case["candidates"]
        relevant_ids = case["relevant_ids"]

        pool_ids = [str(c["id"]) for c in candidates]

        candidate_hit = int(any(rid in relevant_ids for rid in pool_ids))

        ranked = []

        for c in candidates:
            alpha = config["alpha_harmonic"]

            if config["name"] == "BASELINE_07_03":
                adjusted_eff = c["eff_harmonic"]
            else:
                adjusted_eff = (
                    alpha * c["eff_harmonic"] + (1 - alpha) * c["main_coverage"]
                )

            final_score = (
                config["w_semantic"] * c["semantic_minmax"]
                + config["w_eff"] * adjusted_eff
            )

            ranked.append(
                {
                    **c,
                    "adjusted_eff": float(adjusted_eff),
                    "final_score_eval": float(final_score),
                }
            )

        ranked.sort(
            key=lambda x: (
                -x["final_score_eval"],
                -x["semantic_minmax"],
                str(x["id"]),
            )
        )

        ranked_ids = [str(c["id"]) for c in ranked]

        h5 = hit_at_k(
            ranked_ids,
            relevant_ids,
            5,
        )
        h10 = hit_at_k(
            ranked_ids,
            relevant_ids,
            10,
        )
        p5 = precision_at_k(
            ranked_ids,
            relevant_ids,
            5,
        )
        p10 = precision_at_k(
            ranked_ids,
            relevant_ids,
            10,
        )
        r5 = recall_at_k(
            ranked_ids,
            relevant_ids,
            5,
        )
        r10 = recall_at_k(
            ranked_ids,
            relevant_ids,
            10,
        )
        rr = reciprocal_rank(
            ranked_ids,
            relevant_ids,
        )

        candidate_hits.append(candidate_hit)
        h5s.append(h5)
        h10s.append(h10)
        p5s.append(p5)
        p10s.append(p10)
        r5s.append(r5)
        r10s.append(r10)
        rrs.append(rr)

        best_rank = None
        best_title = None

        for rank, c in enumerate(
            ranked,
            start=1,
        ):
            if str(c["id"]) in relevant_ids:
                best_rank = rank
                best_title = c["title"]
                break

        detail_rows.append(
            {
                "config": config["name"],
                "scraped_title": case["scraped_title"],
                "query_text": case["query_text"],
                "candidate_hit": candidate_hit,
                "best_relevant_rank": best_rank,
                "best_relevant_title": best_title,
                "hit_at_5": h5,
                "hit_at_10": h10,
                "precision_at_5": p5,
                "precision_at_10": p10,
                "recall_at_5": r5,
                "recall_at_10": r10,
                "reciprocal_rank": rr,
                "top10_titles": " | ".join(str(c["title"]) for c in ranked[:10]),
            }
        )

    summary_rows.append(
        {
            "config": config["name"],
            "alpha_harmonic": config["alpha_harmonic"],
            "main_weight_inside_eff": (
                0.0
                if config["name"] == "BASELINE_07_03"
                else 1.0 - config["alpha_harmonic"]
            ),
            "w_semantic": config["w_semantic"],
            "w_eff": config["w_eff"],
            "n_query": len(cases),
            "candidate_hit_rate": float(np.mean(candidate_hits)),
            "candidate_hit_count": int(np.sum(candidate_hits)),
            "hit_at_5": float(np.mean(h5s)),
            "hit_at_5_count": int(np.sum(h5s)),
            "hit_at_10": float(np.mean(h10s)),
            "hit_at_10_count": int(np.sum(h10s)),
            "precision_at_5_macro": float(np.mean(p5s)),
            "precision_at_10_macro": float(np.mean(p10s)),
            "recall_at_5_macro": float(np.mean(r5s)),
            "recall_at_10_macro": float(np.mean(r10s)),
            "mrr": float(np.mean(rrs)),
        }
    )


# ============================================================
# 6. Save & print
# ============================================================
summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values(
        [
            "hit_at_5",
            "hit_at_10",
            "mrr",
            "recall_at_5_macro",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

detail_df = pd.DataFrame(detail_rows)

summary_df.to_csv(
    SUMMARY_OUT,
    index=False,
    encoding="utf-8-sig",
)

detail_df.to_csv(
    DETAIL_OUT,
    index=False,
    encoding="utf-8-sig",
)


print("\n" + "=" * 115)
print("HASIL MAIN-INGREDIENT-AWARE EFFICIENCY - VALIDATION ONLY")
print("=" * 115)

print(summary_df.to_string(index=False))

best = summary_df.iloc[0]

baseline = summary_df[summary_df["config"] == "BASELINE_07_03"].iloc[0]

print("\nKONFIGURASI TERBAIK VALIDATION:")
print(
    f"{best['config']} | "
    f"Hit@5={best['hit_at_5']:.4f} "
    f"({int(best['hit_at_5_count'])}/{int(best['n_query'])}) | "
    f"Hit@10={best['hit_at_10']:.4f} | "
    f"Recall@5={best['recall_at_5_macro']:.4f} | "
    f"MRR={best['mrr']:.4f}"
)

print("\nBASELINE:")
print(
    f"Hit@5={baseline['hit_at_5']:.4f} | "
    f"Hit@10={baseline['hit_at_10']:.4f} | "
    f"Recall@5={baseline['recall_at_5_macro']:.4f} | "
    f"MRR={baseline['mrr']:.4f}"
)

print("\nPENTING:")
print("- Candidate retrieval tidak diubah.")
print("- Ground truth hanya dipakai untuk evaluasi.")
print("- Jika main-aware tidak mengalahkan baseline, jangan tambahkan ke production.")
print("- Test 90 query lama tetap tidak dipakai untuk memilih konfigurasi baru.")

print("\nOUTPUT:")
print(f"- {SUMMARY_OUT}")
print(f"- {DETAIL_OUT}")

c:\Users\Mayceren\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


VALIDASI DATA
Jumlah query              : 38
Duplicate query+relevance : 0


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5000.19it/s]



BASE RETRIEVAL CONFIG
Semantic Top-K            : 50
Keyword extra max         : 20
Semantic normalization    : min-max

Membangun candidate pool validation...
Candidate cache selesai.

HASIL MAIN-INGREDIENT-AWARE EFFICIENCY - VALIDATION ONLY
        config  alpha_harmonic  main_weight_inside_eff  w_semantic  w_eff  n_query  candidate_hit_rate  candidate_hit_count  hit_at_5  hit_at_5_count  hit_at_10  hit_at_10_count  precision_at_5_macro  precision_at_10_macro  recall_at_5_macro  recall_at_10_macro      mrr
MAIN050_W07_03            0.50                    0.50         0.7    0.3       38            0.736842                   28  0.368421              14   0.394737               15              0.078947               0.044737           0.328947            0.368421 0.207554
MAIN075_W07_03            0.75                    0.25         0.7    0.3       38            0.736842                   28  0.342105              13   0.421053               16              0.073684               